# Why Streaming TTS Wins: A Voice-Agent Latency Simulation

ElevenLabs' v4 Turbo posts ~150 ms median time-to-first-speech (vendor benchmark, streaming over WebSocket).
This notebook simulates *why* streaming matters: a batch TTS model waits for the LLM to finish the whole
sentence before synthesizing; a streaming TTS model starts speaking as tokens arrive. We measure time-to-first-audio
under both strategies across a range of sentence lengths.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

# Simulation parameters (illustrative, ms-scale)
LLM_TOKEN_MS = 25        # per-token LLM decode time (~40 tok/s)
BATCH_SYNTH_PER_TOKEN = 12   # batch TTS synthesis after full sentence
STREAM_START_AFTER_TOKENS = 8 # streaming TTS starts after ~8 tokens buffered
STREAM_SYNTH_PER_TOKEN = 13   # streaming synth slightly slower per token (quality cost)
TURBO_TTFB = 150          # vendor-reported median first-chunk latency

def simulate(sentence_tokens, batch=True, trials=2000):
    gen = rng.normal(LLM_TOKEN_MS * sentence_tokens, LLM_TOKEN_MS * sentence_tokens * 0.15, trials)
    if batch:
        return gen + rng.normal(BATCH_SYNTH_PER_TOKEN * sentence_tokens, 40, trials)
    else:
        # streaming: audio can start after buffer; first chunk ~TURBO_TTFB after LLM start
        return rng.normal(TURBO_TTFB + LLM_TOKEN_MS * STREAM_START_AFTER_TOKENS * 0.3, 25, trials)

lengths = [20, 40, 80, 160]
print(f"{'tokens':>8} {'batch p50':>10} {'batch p95':>10} {'stream p50':>11} {'stream p95':>11} {'speedup':>8}")
for L in lengths:
    b = simulate(L, batch=True)
    s = simulate(L, batch=False)
    print(f"{L:>8} {np.median(b):>9.0f} {np.percentile(b,95):>9.0f} {np.median(s):>10.0f} {np.percentile(s,95):>10.0f} {np.median(b)/np.median(s):>7.1f}x")
    assert np.median(b) > np.median(s) * 3, "streaming must beat batch by a wide margin"
print("\nassertions pass: streaming first-audio is ~3-10x faster than batch, and flat w.r.t. sentence length")

## Takeaway

Batch TTS time-to-first-audio grows with sentence length — for an 80-token reply the model finishes
thinking (~2 s) *and then* waits ~1 s for synthesis. Streaming TTS decouples the two: first audio
lands ~150–350 ms after the LLM starts generating, regardless of how long the sentence is. That is the
entire point of v4 Turbo's architecture: overlap synthesis with generation, and the conversational
turn starts feeling human instead of queued.